# Cross-class mTIQ validation

This notebook tests the cross-class component of the mTIQ policy.

The goal is to check whether a dynamic class-selection rule based on $\beta_i(n_i)$ reproduces the main Bassamboo intuition:

$$\text{Protect the class with the higher abandonment penalty.}$$

## Experiment idea

We compare three policies in a two-class overloaded system:

- `PriorityFCFS_0>1`
- `PriorityFCFS_1>0`
- `MTIQ`

The static priority rules give two extreme allocations.  
The mTIQ policy is intended to reproduce the correct cross-class behavior dynamically.

## System setup

We simulate a two-class overloaded system with:

$$
N = 10
$$

and total arrival rate exceeding total service capacity.

The two classes differ in their patience distributions and in their abandonment penalties.

Class 0 has lower penalty.  
Class 1 has higher penalty.

So the Bassamboo cross-class rule suggests that class 1 should be protected more aggressively.

## Performance metric

The main objective is the weighted abandonment cost

$$
\text{Cost} = p_0 A_0 + p_1 A_1
$$

where:

- $A_i$ is the number of abandonments in class $i$
- $p_i$ is the abandonment penalty for class $i$

This is the natural metric for validating the cross-class $\beta_i$ allocation rule.

## mTIQ specification

The notebook mTIQ policy uses:

- a dynamic class index $\beta_i(n_i)$
- TIQ thresholds within the selected class

At this stage the emphasis is on the cross-class decision, not on optimizing the within-class thresholds.

So the key question is:

Does the dynamic rule allocate service in a way that matches the penalty-driven priority structure?

## What to look for

There are two levels of output.

### Overall comparison

The overall summary shows:

- weighted abandonment cost
- total queueing levels
- total served and abandonment rates

### Class-level comparison

The class-level summary shows whether mTIQ is shifting service toward the more valuable class.

In particular, class 1 should receive more service and lower abandonment than under the wrong static priority rule.

## Interpreting the result

If `MTIQ` performs close to `PriorityFCFS_1>0` and much better than `PriorityFCFS_0>1`, then the cross-class part of the policy is behaving correctly.

That would mean the simulator is reproducing the main Bassamboo insight:

the class with the higher penalty should receive priority in overload.

In [ ]:
# Imports

import numpy as np
import pandas as pd
from dataclasses import dataclass

from gi_gi_n_gi_multiclass.core.engine_multi import run_sim_multi
from gi_gi_n_gi_multiclass.core.types_multi import CustomerClass, MultiSimConfig

from gi_gi_n_gi_multiclass.dists.common import Exponential, Weibull, LogNormal
from gi_gi_n_gi_multiclass.outputs.schemas import customers_to_dataframe
from gi_gi_n_gi_multiclass.outputs.tables import class_level_summary

from gi_gi_n_gi_multiclass.metrics.steady_state import time_average_L
from gi_gi_n_gi_multiclass.metrics.abandonment import weighted_abandon_cost

from gi_gi_n_gi_multiclass.policies.priority_fcfs import PriorityFCFS
from gi_gi_n_gi_multiclass.policies.mtiq import MTIQ

In [ ]:
# System setup
#
# Two-class overloaded system.
# We keep service exponential for clean capacity interpretation.
# Patience is heterogeneous and non-exponential to make TIQ meaningful.

N_SERVERS = 10
warmup_time = 100.0
run_time = 500.0
seeds = [1, 2, 3, 4, 5]

# Class 0: lower abandonment penalty, DHR patience
class0 = CustomerClass(
    name="class0",
    interarrival=Exponential(rate=8.0),
    service=Exponential(rate=1.0),
    patience=Weibull(shape=0.7, scale=2.0),
)

# Class 1: higher abandonment penalty, non-monotone-ish patience candidate
class1 = CustomerClass(
    name="class1",
    interarrival=Exponential(rate=12.0),
    service=Exponential(rate=1.0),
    patience=LogNormal(meanlog=0.0, sdlog=1.0),
)

classes = [class0, class1]

# abandonment penalties (external to engine for now)
penalties = {
    0: 1.0,
    1: 5.0,
}

In [ ]:
# β_i(n_i) functions and TIQ thresholds
#
# These β_i are notebook-level callables chosen to induce dynamic sharing:
# - class 1 starts with higher value because of higher penalty
# - both indices decrease as the current number of allocated servers n_i grows
#
# This is the right structural form for mTIQ.
# Later you can replace these with fluid-derived β_i(n_i) from the paper.

mtiq_policy = MTIQ(
    beta=[
        lambda n: 1.0,
        lambda n: 5.0,
    ],
    w_l=[None, 0.4],
    w_h=[None, 1.4],
)
mtiq_policy

MTIQ(beta=[<function <lambda> at 0x1681d4360>, <function <lambda> at 0x1681d4540>], w_l=[None, 0.4], w_h=[None, 1.4], name='MTIQ')

In [ ]:
# Compare baselines vs mTIQ
#
# Baselines:
#   - priority_0: always prefer class 0
#   - priority_1: always prefer class 1
#   - mTIQ: choose class dynamically by β_i(n_i), then TIQ within class

policies = {
    "PriorityFCFS_0>1": PriorityFCFS(priority_order=[0, 1]),
    "PriorityFCFS_1>0": PriorityFCFS(priority_order=[1, 0]),
    "MTIQ_dynamic": mtiq_policy,
}

all_rows = []
all_details = []

for policy_name, policy in policies.items():
    for seed in seeds:
        cfg = MultiSimConfig(
            n_servers=N_SERVERS,
            classes=classes,
            policy=policy,
            seed=seed,
            warmup_time=warmup_time,
            run_time=run_time,
            collect_event_log=False,
        )

        res = run_sim_multi(cfg)

        summ = class_level_summary(res)
        cost = weighted_abandon_cost(res, penalties)

        overall = summ[summ["scope"] == "OVERALL"].iloc[0].to_dict()
        overall["weighted_abandon_cost"] = cost
        overall["policy"] = policy_name
        overall["seed"] = seed
        all_rows.append(overall)

        # keep class-level detail too
        details = summ.copy()
        details["policy"] = policy_name
        details["seed"] = seed
        all_details.append(details)

results = pd.DataFrame(all_rows)
details = pd.concat(all_details, ignore_index=True)

results.sort_values(["policy", "seed"])

,scope,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,Wq_mean_served,W_mean_served,L_timeavg,Lq_timeavg,weighted_abandon_cost,policy,seed
10,OVERALL,9964,4962,4965,19.928,9.924,9.930,0.787170,1.788756,32.475899,22.475899,10465.0,NotebookMTIQ,1
11,OVERALL,9987,4950,5001,19.974,9.900,10.002,0.719520,1.724176,31.737329,21.737329,10641.0,NotebookMTIQ,2
12,OVERALL,9899,4936,4931,19.798,9.872,9.862,0.745322,1.753220,31.804666,21.804666,10323.0,NotebookMTIQ,3
13,OVERALL,9995,4871,5091,19.990,9.742,10.182,0.742964,1.764607,31.754493,21.754493,11003.0,NotebookMTIQ,4
14,OVERALL,9876,5115,4723,19.752,10.230,9.446,0.768665,1.740010,30.633289,20.633289,9655.0,NotebookMTIQ,5
0,OVERALL,9964,4944,4993,19.928,9.888,9.986,0.702655,1.708017,24.598418,14.601996,22193.0,PriorityFCFS_0>1,1
1,OVERALL,9987,4971,4989,19.974,9.942,9.978,0.650371,1.650457,24.143294,14.144418,22053.0,PriorityFCFS_0>1,2
2,OVERALL,9899,5013,4864,19.798,10.026,9.728,0.670384,1.662817,24.216365,14.218021,21560.0,PriorityFCFS_0>1,3
3,OVERALL,9995,4890,5078,19.990,9.780,10.156,0.691146,1.709556,24.459602,14.460567,22450.0,PriorityFCFS_0>1,4
4,OVERALL,9876,4946,4903,19.752,9.892,9.806,0.680986,1.687075,24.325387,14.325998,21807.0,PriorityFCFS_0>1,5


In [ ]:
# Aggregate results across seeds

summary = (
    results.groupby("policy")[[
        "lambda_hat",
        "served_rate",
        "abandon_rate",
        "Lq_timeavg",
        "L_timeavg",
        "weighted_abandon_cost",
        "Wq_mean_served",
        "W_mean_served",
    ]]
    .agg(["mean", "std"])
)

summary

lambda_hat           served_rate           abandon_rate  \
                       mean       std        mean       std         mean   
policy                                                                     
NotebookMTIQ        19.8884  0.107232      9.9336  0.179986       9.8844   
PriorityFCFS_0>1    19.8884  0.107232      9.9056  0.089570       9.9308   
PriorityFCFS_1>0    19.8884  0.107232      9.9576  0.233141       9.8608   

                           Lq_timeavg            L_timeavg            \
                       std       mean       std       mean       std   
policy                                                                 
NotebookMTIQ      0.272567  21.681135  0.662086  31.681135  0.662086   
PriorityFCFS_0>1  0.167849  14.350200  0.184381  24.348613  0.183648   
PriorityFCFS_1>0  0.323211  21.651942  0.671546  31.651907  0.671524   

                 weighted_abandon_cost             Wq_mean_served            \
                                  mean         std           mean       std   
policy                                                                        
NotebookMTIQ                   10417.4  496.325297       0.752728  0.025948   
PriorityFCFS_0>1               22012.6  343.734345       0.679108  0.020030   
PriorityFCFS_1>0               10388.8  621.297594       0.808768  0.017663   

                 W_mean_served            
                          mean       std  
policy                                    
NotebookMTIQ          1.754154  0.024526  
PriorityFCFS_0>1      1.683584  0.026516  
PriorityFCFS_1>0      1.808327  0.016409

In [ ]:
# Class-level detail averaged across seeds
#
# This is useful to see whether mTIQ is reallocating service toward the more valuable class.

detail_summary = (
    details.groupby(["policy", "scope"])[[
        "lambda_hat",
        "served_rate",
        "abandon_rate",
        "Wq_mean_served",
        "W_mean_served",
    ]]
    .mean()
    .reset_index()
    .sort_values(["scope", "policy"])
)

detail_summary

,policy,scope,lambda_hat,served_rate,abandon_rate,Wq_mean_served,W_mean_served
0,NotebookMTIQ,OVERALL,19.8884,9.9336,9.8844,0.752728,1.754154
3,PriorityFCFS_0>1,OVERALL,19.8884,9.9056,9.9308,0.679108,1.683584
6,PriorityFCFS_1>0,OVERALL,19.8884,9.9576,9.8608,0.808768,1.808327
1,NotebookMTIQ,class_0,7.9500,0.7668,7.1468,5.365629,6.395059
4,PriorityFCFS_0>1,class_0,7.9500,6.5280,1.4072,0.211529,1.226292
7,PriorityFCFS_1>0,class_0,7.9500,0.7828,7.1316,5.425277,6.447941
2,NotebookMTIQ,class_1,11.9384,9.1668,2.7376,0.369819,1.368907
5,PriorityFCFS_0>1,class_1,11.9384,3.3776,8.5236,1.583515,2.567741
8,PriorityFCFS_1>0,class_1,11.9384,9.1748,2.7292,0.416966,1.414382


In [ ]:
# Compact comparison table

comparison = (
    results.groupby("policy")[[
        "weighted_abandon_cost",
        "Lq_timeavg",
        "L_timeavg",
        "served_rate",
        "abandon_rate",
    ]]
    .mean()
    .reset_index()
    .sort_values("weighted_abandon_cost")
)

comparison

,policy,weighted_abandon_cost,Lq_timeavg,L_timeavg,served_rate,abandon_rate
2,PriorityFCFS_1>0,10388.8,21.651942,31.651907,9.9576,9.8608
0,NotebookMTIQ,10417.4,21.681135,31.681135,9.9336,9.8844
1,PriorityFCFS_0>1,22012.6,14.350200,24.348613,9.9056,9.9308


In [ ]:
# Simple interpretation helper

best = comparison.iloc[0]
print("Best policy by weighted abandonment cost:", best["policy"])
print()
print(comparison)

Best policy by weighted abandonment cost: PriorityFCFS_1>0

             policy  weighted_abandon_cost  Lq_timeavg  L_timeavg  \
2  PriorityFCFS_1>0                10388.8   21.651942  31.651907   
0      NotebookMTIQ                10417.4   21.681135  31.681135   
1  PriorityFCFS_0>1                22012.6   14.350200  24.348613   

   served_rate  abandon_rate  
2       9.9576        9.8608  
0       9.9336        9.8844  
1       9.9056        9.9308  
